### Q1 — What is overall sales performance?
Top-line KPIs: revenue, orders, units, AOV, gross profit, margin.

In [0]:
%sql
SELECT
  COUNT(DISTINCT order_id)                              AS completed_orders,
  SUM(quantity)                                         AS total_units_sold,
  ROUND(SUM(item_revenue), 2)                           AS total_revenue,
  ROUND(SUM(gross_profit), 2)                           AS total_gross_profit,
  ROUND(SUM(gross_profit) / NULLIF(SUM(item_revenue), 0) * 100, 2) AS gross_margin_pct,
  ROUND(SUM(item_revenue) / NULLIF(COUNT(DISTINCT order_id), 0), 2) AS avg_order_value,
  ROUND(SUM(item_revenue) / NULLIF(SUM(quantity), 0), 2) AS avg_revenue_per_unit
FROM ecommerce.mart.fct_sales_flat
WHERE is_completed = TRUE;

### Q2 — How is sales changing over time?
Monthly revenue, orders, units, and month-over-month growth.

In [0]:
%sql
WITH monthly AS (
  SELECT
    year_month,
    MIN(year)  AS year,
    MIN(month) AS month,
    COUNT(DISTINCT order_id)     AS orders_count,
    SUM(quantity)                AS units_sold,
    ROUND(SUM(item_revenue), 2)  AS revenue,
    ROUND(SUM(gross_profit), 2)  AS gross_profit
  FROM ecommerce.mart.fct_sales_flat
  WHERE is_completed = TRUE
  GROUP BY year_month
)
SELECT
  year_month,
  orders_count,
  units_sold,
  revenue,
  gross_profit,
  ROUND(revenue - LAG(revenue) OVER (ORDER BY year, month), 2)                       AS revenue_mom_change,
  ROUND((revenue - LAG(revenue) OVER (ORDER BY year, month))
        / NULLIF(LAG(revenue) OVER (ORDER BY year, month), 0) * 100, 2)              AS revenue_mom_pct
FROM monthly
ORDER BY year, month;

### Q3 — Which periods are strongest / weakest?
Three cuts: by month-of-year (seasonality), by weekday vs weekend, and by holiday flag. Ranked so strongest/weakest are easy to read off.

In [0]:
%sql
-- 3a: Seasonality — revenue by calendar month name, aggregated across all years
SELECT
  month,
  month_name,
  COUNT(DISTINCT order_id)    AS orders_count,
  ROUND(SUM(item_revenue), 2) AS revenue,
  RANK() OVER (ORDER BY SUM(item_revenue) DESC) AS revenue_rank
FROM ecommerce.mart.fct_sales_flat
WHERE is_completed = TRUE
GROUP BY month, month_name
ORDER BY revenue DESC;

In [0]:
%sql
-- 3b: Weekday vs weekend performance
SELECT
  is_weekend,
  COUNT(DISTINCT order_id)                                  AS orders_count,
  ROUND(SUM(item_revenue), 2)                               AS revenue,
  ROUND(SUM(item_revenue) / NULLIF(COUNT(DISTINCT order_id), 0), 2) AS avg_order_value
FROM ecommerce.mart.fct_sales_flat
WHERE is_completed = TRUE
GROUP BY is_weekend;

In [0]:
%sql
-- 3c: Holiday lift — do flagged holiday dates outperform an average day?
SELECT
  is_holiday,
  holiday_name,
  COUNT(DISTINCT order_date)                                        AS days_covered,
  ROUND(SUM(item_revenue), 2)                                       AS revenue,
  ROUND(SUM(item_revenue) / NULLIF(COUNT(DISTINCT order_date), 0), 2) AS avg_revenue_per_day
FROM ecommerce.mart.fct_sales_flat
WHERE is_completed = TRUE AND is_holiday = TRUE
GROUP BY is_holiday, holiday_name
ORDER BY avg_revenue_per_day DESC;

### Q4 — Which products generate the most revenue?


In [0]:
%sql
SELECT
  product_id,
  product_name,
  brand,
  category_name,
  SUM(quantity)                                          AS units_sold,
  ROUND(SUM(item_revenue), 2)                            AS revenue,
  ROUND(SUM(gross_profit), 2)                            AS gross_profit,
  ROUND(SUM(gross_profit) / NULLIF(SUM(item_revenue), 0) * 100, 2) AS gross_margin_pct,
  RANK() OVER (ORDER BY SUM(item_revenue) DESC)          AS revenue_rank
FROM ecommerce.mart.fct_sales_flat
WHERE is_completed = TRUE
GROUP BY product_id, product_name, brand, category_name
ORDER BY revenue DESC
LIMIT 20;